# Train all emotion models on a Colab GPU

Runtime > Change runtime type > **T4 GPU** before running.

Upload `colab_package.zip` to `MyDrive/emotion_2b/` first. Results are written back to the same Drive folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/emotion_2b'
!ls -la $DRIVE

In [ ]:
!rm -rf /content/project && mkdir -p /content/project
!unzip -q $DRIVE/colab_package.zip -d /content/project
%cd /content/project
!ls
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Colab already has torch, transformers, scikit-learn, pandas and matplotlib
import torch; print(torch.__version__, 'GPU:', torch.cuda.is_available())

In [ ]:
# Quick check (about 1-2 min): every model for 1 epoch + all test/plot code, in a throw-away copy
!rm -rf /content/smoke && cp -r /content/project /content/smoke
%cd /content/smoke
!python -W ignore -c "import sys; sys.path.insert(0,'.'); import src.pipeline as P; P.MAX_EPOCHS=1; [P.train(v,s) for v in P.VARIANTS for s in P.SEEDS]"
!python -W ignore models/speech_pipeline/test.py && python -W ignore models/text_pipeline/test.py && python -W ignore models/fusion_pipeline/test.py && echo SMOKE TEST OK
%cd /content/project

In [ ]:
!python -W ignore models/text_pipeline/train.py

In [ ]:
!python -W ignore models/speech_pipeline/train.py

In [ ]:
!python -W ignore models/fusion_pipeline/train.py

In [ ]:
!python -W ignore models/speech_pipeline/test.py
!python -W ignore models/text_pipeline/test.py
!python -W ignore models/fusion_pipeline/test.py

In [ ]:
# save checkpoints and results back to Drive
!zip -q -r $DRIVE/colab_results.zip checkpoints Results
!ls -la $DRIVE